# A Corpus Query Language of Your Own

**Computational Text Analysis Pack · Module 15 — Rule-Based and Symbolic Methods**

*Technique:* CQL/CQP semantics; querying token sequences rather than characters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/04-rule-based-methods/15b_corpus_query_language.ipynb)

Data: `masakhanews.csv`, `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv', 'colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A regular expression matches characters. A **corpus query language** matches
*tokens*, with attributes — word form, lemma, part of speech — and that
difference is what lets a corpus linguist ask a grammatical question instead of
a string question.

`[word="the"] [pos="ADJ"]{1,3} [lemma="king"]` says: *the*, then one to three
adjectives, then any form of *king*. No regex over raw text can express that,
because the units are wrong.

We are going to implement enough of CQL to run queries like that.

## The tooling

**CQL/CQP** comes from the IMS **Open Corpus Workbench (CWB)** and is the
lingua franca: **CQPweb** (Hardie), **Sketch Engine** and **BlackLab** all speak
it. **Tregex/Tsurgeon** does the equivalent job over syntax trees.

None of them are installed, and the point of writing our own is that CQL's
semantics are simple enough to implement in a page — the industrial versions win
on indexing and scale, not on expressiveness.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import re

# A token here carries attributes, exactly as in a CWB-indexed corpus. We have
# no tagger, so `pos` is a crude rule-based guess — enough to show the mechanism.
FUNCTION = {"the", "a", "an", "of", "in", "to", "and", "is", "was", "for",
            "on", "with", "at", "by", "from", "that", "this", "it"}

def tag(word):
    if word in FUNCTION:
        return "FUNC"
    if word.endswith(("ly",)):
        return "ADV"
    if word.endswith(("ing", "ed")):
        return "VERB"
    if word.endswith(("ous", "ful", "ive", "al", "ic")):
        return "ADJ"
    if word.endswith("s"):
        return "NOUNP"
    return "NOUN"

def to_tokens(text):
    return [{"word": w, "pos": tag(w)} for w in tokenise(text)]

docs = [to_tokens(t) for t in df_colonial["text"].head(1200)]
print(f"{len(docs)} documents, {sum(len(d) for d in docs):,} tokens, "
      f"attributes: word, pos")

## Parsing the query language

A CQL query is a sequence of token constraints, each optionally with a
repetition range. That is the whole grammar we need:

- `[word="king"]` — attribute equals value
- `[pos="ADJ"]{1,3}` — one to three adjectives
- `[]` — any token
- `[word="king|queen"]` — alternation, since values are regexes

In [ ]:
TOKEN_RE = re.compile(r'\[([^\]]*)\](?:\{(\d+),(\d+)\})?')

def parse_cql(query):
    steps = []
    for body, lo, hi in TOKEN_RE.findall(query):
        constraints = {}
        for attr, val in re.findall(r'(\w+)\s*=\s*"([^"]*)"', body):
            constraints[attr] = re.compile(f"^(?:{val})$")
        steps.append({"constraints": constraints,
                      "min": int(lo) if lo else 1,
                      "max": int(hi) if hi else 1})
    return steps

def token_matches(tok, constraints):
    return all(rx.match(tok.get(attr, "")) for attr, rx in constraints.items())

def match_at(tokens, i, steps):
    """Greedy-with-backtracking match of `steps` starting at position i."""
    if not steps:
        return i
    step, rest = steps[0], steps[1:]
    n = 0
    while n < step["max"] and i + n < len(tokens) and \
            token_matches(tokens[i + n], step["constraints"]):
        n += 1
    for take in range(n, step["min"] - 1, -1):
        end = match_at(tokens, i + take, rest)
        if end is not None:
            return end
    return None

def cql_search(docs, query, limit=15):
    steps = parse_cql(query)
    hits = []
    for d_i, toks in enumerate(docs):
        for i in range(len(toks)):
            end = match_at(toks, i, steps)
            if end is not None and end > i:
                hits.append((d_i, i, " ".join(t["word"] for t in toks[i:end])))
                if len(hits) >= limit:
                    return hits
    return hits

print("query engine ready")

In [ ]:
for q in ['[word="the"] [pos="ADJ"]{1,2} [pos="NOUN"]',
          '[word="king|queen|chief"] [word="of"] []',
          '[pos="ADV"] [pos="VERB"]']:
    print(f"\nCQL: {q}")
    for _, _, phrase in cql_search(docs, q, limit=6):
        print(f"   {phrase}")

## Why this beats grep

The test is a question that is trivial in CQL and effectively impossible as a
string search: *find every noun phrase introduced by* the *with at least two
adjectives*. The pattern is about grammatical structure, and the tokens carry
that structure.

In [ ]:
q = '[word="the"] [pos="ADJ"]{2,3} [pos="NOUN|NOUNP"]'
hits = cql_search(docs, q, limit=200)
print(f"{len(hits)} matches for: {q}\n")
heads = Counter(h[2].split()[-1] for h in hits)
print("most common head nouns in that construction:")
for w, n in heads.most_common(12):
    print(f"   {w:<16}{n:>4}")
print()
print("This is a construction-level frequency list. It is the kind of object")
print("corpus linguistics is built on, and it does not exist at the string level.")

In [ ]:
# The same engine over the multilingual news corpus, to show the limit.
news = [to_tokens(t) for t in df[df["language"] == "Hausa"]["text"].head(400)]
print("Hausa, querying by word works:")
for _, _, p in cql_search(news, '[word="gwamnati.*"] []', limit=5):
    print("   ", p)
print()
print("Querying by pos does NOT work: our `tag` function is a set of English")
print("suffix rules. In CWB you would index a corpus tagged by a real tagger for")
print("that language, and for most African languages that tagger is the missing")
print("piece — not the query engine.")

### 🔧 Your turn 1

Write a query for a prepositional phrase — a function word, then up to two adjectives, then a noun — and compare its head nouns against the ones found above.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The head-noun distributions differ sharply, which is the payoff: *the* + ADJ +
NOUN pulls out the corpus's descriptive vocabulary, while a prepositional frame
pulls out its spatial and relational vocabulary. Running both and comparing is a
construction-based method, in the Biber lexical-bundles tradition, and it is
only available once your unit of search is the token rather than the character.
Be aware that our `pos` tagger is rules-on-suffixes and wrong often; every
number here inherits that error, which is why module 16 builds a real tagger.

</details>